In deze notebook wordt de feature uitgewerkt om de content in huisstijl te uploaden (DFIT-1261).

Om de content opmaak aan te passen maken we eerst een nieuw topic aan, in dit geval een stap. Na het runnen van onderstaande code kun je in cms zien dat content inderdaad in huisstijl is opgemaakt.

In [1]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.workflow import Workflow
from ask_delphi_api.import_digicoach import Import
from ask_delphi_api.authentication import AskDelphiClient

client = AskDelphiClient()
project = Project(client)
topics = TopicTools(client, project)
workflow = Workflow(client)
import_service = Import()


AskDelphi Client loaded.
Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


In [ ]:
# voor een stap, vul je "Action" als tweede variabel in
topicId = topics.topic_upload("Nieuwe stap testje2", "Action")
text = f'<h1>Titel</h1><p>Inleiding met <strong>vet</strong> tekst.</p><ul><li>Eerste punt</li><li>Tweede punt</li></ul>'
result = topics.checkout(topicId)
topicVersionId = result['topicVersionId']

import_service.add_content_to_topic(topicId, topicVersionId, text)

# Checkin topic.
topics.checkin(topicId)

# Topic publiceren
request_id = workflow.create_workflow_transition_request(topicId)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)

Hieronder gaan we onderzoeken of het ook mogelijk is om een taaksjabloon uit een word document te kunnen uploaden waarbij de opmaak van de content wordt meegenomen.

In [2]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
from pathlib import Path

project_root = Path.cwd().parent
paths = read_dir(project_root/'Taaksjablonen')

json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_coach = convert_doc_to_json(path)
        json_coaches.append(json_coach)
    except ValueError as e:
        print(e)

LIC - Taaksjablonen Sanering v1.0.docx



retrieved 9 tables from doc c:\Users\veldi01\Documents\git\cog-boa-digicoaches\ask-delphi-api\Taaksjablonen\LIC - Taaksjablonen Sanering v1.0.docx
Found title: Sanering (LIC)
Found tag: Directie CAP
Found tag: Keten Inning en betalingsverkeer
Found sources: [{'titel': 'Kwijtscheldingsformulier Ondernemers', 'type': 'Bronnen', 'link': 'https://www.belastingdienst.nl/wps/wcm/connect/bldcontentnl/themaoverstijgend/programmas_en_formulieren/kwijtschelding_van_belasting_enof_premie_voor_ondernemingen'}, {'titel': 'Applicatie overzicht', 'type': 'Applicaties', 'link': 'https://connectpeople.belastingdienst.nl/files/form/anonymous/api/library/1fa8a980-ab6d-4c9a-a059-da1fea2c3dbf/document/1a372d7d-d07f-448a-8f2b-efa5c9690ec8/media/6980%20LIC%20-%20applicatie%20overzicht.pdf'}, {'titel': 'GDC - Standaardzinnen voor de sanering', 'type': 'Handleidingen en instructies', 'link': 'Word document (zelfde link als in de DC Sanering (Taskforce)'}, {'titel': 'LI

In [4]:
import json
json_digicoach = json.loads(json_coaches[0])
taken = json_digicoach["tasks"]

for taak in taken:
    print(taak["name"])

# steps = taken[1]["steps"]
# print(taken)


Selecteer Zaak
Quick scan verzoek
verzoek bekijken
Verzoek beoordelen
Beschikking Uitbrengen


In [ ]:
taken = json_digicoach["tasks"]
print(taken[1])

Hieronder selecteer ik 1 willekeurige taak uit de json en zet deze als topic in het cms

In [ ]:
# Haal takenlijst uit json_digicoach
topic_id_list = []
taken = json_digicoach["tasks"]

taken = taken[1]
# Voor elke taak, maak topic
for taak in taken:
    topic_id_task = topics.topic_upload(taak[0], "Action")
    topic_id_list.append(topic_id_task)
    result = topics.checkout(topic_id_task)
    topic_version_id_task = result['topicVersionId']
    # Maak topic met naam taak uit json
    ### Task topic aanmaken en daaronder de steps hangen

    # Voeg beschrijving als content toe
    import_service.add_content_to_topic(topic_id_task, topic_version_id_task, taak["description"])

    # Haal stappenlijst uit taak
    steps = taak["steps"]

    for step in steps:
    # Maak topic met naam step uit json
        topic_id_step, topic_version_id_step = import_service.create_step(step["name"], topic_id_task, topic_version_id_task)
        topic_id_list.append(topic_id_step)
        # Toevoegen beschrijving aan de topic content
        import_service.add_content_to_topic(topic_id_step, topic_version_id_step, step["description"])

        # Aanwezige links toevoegen aan pyramide
        #import_service.add_sources(topic_id_task, topic_version_id_task, step[1], sources)

        import_service._get_topic().checkin(topic_id_step)

    # Inchecken taak
    import_service._get_topic().checkin(topic_id_task)

#import_service._get_topic().checkin(topic_id_digicoach)
#import_service._get_topic().checkin(topic_id_predefined_search)

for topic_id in topic_id_list:
    import_service.publiceer(topic_id)
